In [1]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np
import asyncio
import aiohttp

In [2]:
class recipe_fetch():
    def __init__(self):
        self.conn = sql.connect_pc()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.conn.close()
        self.conn = None

    def __del__(self):
        if self.conn:
            self.conn.close()


    def id_range(self):
        cursor = self.conn.cursor()
        query = """
        SELECT recipe_id FROM recipes
        WHERE name LIKE '%Charm%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ingot%'
        OR name LIKE '%Powder%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ghostdust%'
        OR name LIKE '%Gold Coin%'
        OR name LIKE '%Potion%'
        OR name LIKE '%Fangs%'
        OR name LIKE '%Campfire%'
        OR name LIKE '%Woodsman%'
        OR name LIKE '%Intricate%'
        OR name LIKE '%LockPicks%'
        OR name LIKE '%Coinbag%'
        """
        # query = """
        # SELECT recipe_id FROM recipes
        # """
        cursor.execute(query)
        return cursor.fetchall()

    def craftable(self, recipe_id):
        cursor = self.conn.cursor()
        query = f"""
        SELECT amount, rarity, name FROM recipes
        WHERE recipe_id = {recipe_id}
        """
        cursor.execute(query)
        return cursor.fetchall() 

    def ingredients(self, recipe_id):
        cursor = self.conn.cursor()
        query = f"""
        SELECT amount, rarity, name FROM ingredients
        WHERE recipe_id = {recipe_id}
        """
        cursor.execute(query)
        return cursor.fetchall()

In [3]:
class darkerdb():
    def __init__(self):
        self.from_date = (datetime.now(timezone.utc) - timedelta(minutes=10)).strftime("%Y-%m-%dT%H:%M:%SZ")

    async def r_fetch(self, session, amount, rarity, name):
        url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "’")}&rarity={rarity}&from={self.from_date}&limit=50"
        response = await session.get(url)
        output = await response.json()
        response.release()

        body_data = output['body']
        price_per_unit = [x['price_per_unit'] for x in body_data]
        
        q_sold = len(price_per_unit)
        r_avg = int(np.min(price_per_unit) * amount) if price_per_unit else 0
        return name, rarity, r_avg, q_sold


    async def i_fetch(self, session, amount, rarity, name):
        url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "’")}&rarity={rarity}&from={self.from_date}&limit=50"
        response = await session.get(url)
        output = await response.json()
        response.release()

        body_data = output['body']
        price_per_unit = [x['price_per_unit'] for x in body_data]
        
        if name == 'Gold Coin': i_avg = amount
        elif price_per_unit: i_avg = int(np.min(price_per_unit) * amount) 
        else: i_avg = 0

        return name, rarity, i_avg

In [4]:
async def process_fetch(session, r, db, sem_limit, recipe_id):
    async with sem_limit:
        r_task = [db.r_fetch(session, r_amount, r_rarity, r_name) for r_amount, r_rarity, r_name in r.craftable(recipe_id)]
        i_task = [db.i_fetch(session, i_amount, i_rarity, i_name) for i_amount, i_rarity, i_name in r.ingredients(recipe_id)]
        r_results = await asyncio.gather(*r_task)
        i_results = await asyncio.gather(*i_task)

    recipe_id = recipe_id
    rarity = r_results[0][1]        
    name = r_results[0][0]
    r_avg = r_results[0][2]
    i_avg = sum(ingredient[2] for ingredient in i_results)
    net = r_avg - i_avg
    q_sold = r_results[0][3]
    return(recipe_id, rarity, name, net, r_avg, i_avg, q_sold, i_results)


async def main():
    header = [('id', 'rarity', 'name', 'net', 'r_avg', 'i_avg', 'q_sold', 'ingredients / rarity / price')]

    r = recipe_fetch()
    db = darkerdb()
    sem = asyncio.Semaphore(30)
    async with aiohttp.ClientSession() as session:
        task = [process_fetch(session, r, db, sem, recipe_id[0]) for recipe_id in r.id_range()]
        
        return header + await asyncio.gather(*task)

In [5]:
if __name__ == '__main__': 
    output = await main()
    output[1:] = sorted(output[1:], key=lambda x: x[3], reverse=True)
    for x in output:
        print(f"{x[0]:<10} {x[1]:<10} {x[2]:<35} {(x[3]):>8} {x[4]:>8} {x[5]:>8} {x[6]:>8}   {x[7]}")

id         rarity     name                                     net    r_avg    i_avg   q_sold   ingredients / rarity / price
53         Epic       Froststone Ingot                        1389     2499     1110        3   [('Froststone Ore', 'Epic', 1110)]
308        Epic       Froststone Ingot                        1389     2499     1110        3   [('Froststone Ore', 'Epic', 1110)]
15         Epic       Obsidian Powder                         1066     1866      800        1   [('Obsidian Ore', 'Epic', 800)]
67         Epic       Obsidian Ingot                           930     2130     1200        1   [('Obsidian Ore', 'Epic', 1200)]
322        Epic       Obsidian Ingot                           930     2130     1200        1   [('Obsidian Ore', 'Epic', 1200)]
95         Rare       Charm of Fortune                         476     2800     2324        4   [("Cockatrice's Lucky Feather", 'Rare', 1777), ('Frosted Feather', 'Rare', 465), ('Bowstring', 'Common', 32), ('Gold Coin', 'Unique